# Fault slip analysis (FSA) for GA well-placement optimization — Vienna geothermal
Copy of `nb3-2_FSA_stress_based_Vienna.ipynb`, adapted for a GA generation folder (e.g. `sim_gen_000_sim_py`), where each case has a different well layout and sampled parameters.

- Inputs: principal stresses per case (`caseN_STRESMXP/STRESMNP/STRESINT.npy`, shape `(n_is,n_js,n_ks,n_times)`), grid coordinates + fault ID + per-cell dip angle/azimuth (`JD_geothermal_coor&fault&dip.npy`, layer 4 = dip angle, layer 5 = dip azimuth), and the GA case table (`gen_XXX_cases_seedYY.csv`, gives `SH_azi_deg` per case).
- **Per-cell fault orientation**: strike = dip_azimuth − 90 (right-hand rule), as in `nb3-2_FSA_stress_based_Vienna_per_cell_orientation.ipynb`.
- Normal faulting regime: Sv (STRESMXP) > SH (STRESINT) > Sh (STRESMNP).
- Output CSV: `case, fault0, fault1` = 99th percentile (over fault cells) of the shear / effective normal stress ratio, taken at the time step selected by `time_mode`.
- Cases with missing/incomplete stress files (failed simulations) are written as NaN so every case in the GA table appears in the CSV.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import sys

# add src to path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from fault_slip_analysis import FSA_stress_based_per_cell

################## User Inputs ##############################
gen_name = 'gen_000' # GA generation name
n_faults = 2 # number of faults
fault_cohesion = 0 # fault cohesion in kPa
friction_coefficient = 0.6
percentile = 99 # percentile of stress ratio over fault cells
# time step used for the CSV value:
#   'max'  -> max over time steps (excluding the initial step) of the per-time-step percentile
#   'last' -> final time step only
#   int    -> a specific time index
#   'YYYY-MM-DD' -> a specific output date; needs output_dates below.
#                  Cases whose arrays do not reach that date (simulation stopped early) get NaN.
time_mode = 'last'
# output dates of the time axis in the npy files (index 0 = initial step). The npy files store no dates,
# so list them here (from the CMG deck output times or the 'TIME =' headers in the .rwo files).
# output_dates = None # e.g. ['2030-01-01', '2040-01-01', '2050-01-01', '2060-01-01']
output_dates = ['2030-01-01', '2040-01-01', '2050-01-01', '2060-01-01']
# paths to input data
GA_folder = repo_root/'results'/'Vienna_geothermal'/'GA'
cases_csv_path = GA_folder/f'{gen_name}_cases_seed11.csv' # GA case table (case, layout, sampled parameters)
principal_stresses_folder = GA_folder/f'sim_{gen_name}_sim_py' # folder with caseN_STRESMXP/STRESMNP/STRESINT.npy
coor_fault_npy_path = repo_root/'results'/'Vienna_geothermal'/'coor_fault'/'JD_geothermal_coor&fault&dip.npy' # coordinates, fault ID, dip angle, dip azimuth
# output
output_csv_path = GA_folder/f'{gen_name}_FSA_percell_p{percentile}_stress_ratio.csv'
output_csv_per_time_path = GA_folder/f'{gen_name}_FSA_percell_p{percentile}_stress_ratio_per_time.csv'
################## End of User Inputs #######################

# load data (utf-8-sig strips the BOM in the CSV headers)
cases = pd.read_csv(cases_csv_path, encoding='utf-8-sig')
coor_fault = np.load(coor_fault_npy_path)
# per-cell fault geometry (layer 4 = dip angle, layer 5 = dip azimuth); strike = dip_azimuth - 90 (right-hand rule)
fault_masks   = [coor_fault[:,:,:,3] == f for f in range(n_faults)]
fault_dips    = [coor_fault[:,:,:,4][m] for m in fault_masks]
fault_strikes = [(coor_fault[:,:,:,5][m] - 90) % 360 for m in fault_masks]
for f in range(n_faults):
    # circular mean of strike (strikes can wrap around 0/360)
    s_rad = np.radians(fault_strikes[f])
    strike_mean = np.degrees(np.arctan2(np.sin(s_rad).mean(), np.cos(s_rad).mean())) % 360
    strike_dev = (fault_strikes[f] - strike_mean + 180) % 360 - 180
    print(f'fault{f}: dip {fault_dips[f].mean():.2f} +/- {fault_dips[f].std():.2f} deg, '
          f'strike {strike_mean:.2f} +/- {strike_dev.std():.2f} deg')
case_nums = sorted(cases['case'].astype(int).unique())
print(f'{len(case_nums)} cases in {cases_csv_path.name}')
print('fault cells:', [int(m.sum()) for m in fault_masks])

# resolve a date time_mode to a time index
if isinstance(time_mode, str) and time_mode not in ('max', 'last'):
    if output_dates is None:
        raise ValueError('time_mode is a date: set output_dates to the dates of the npy time axis')
    dates = pd.to_datetime(output_dates)
    target = pd.to_datetime(time_mode)
    if target not in dates:
        raise ValueError(f'{time_mode} is not an output date; available: {list(output_dates)}')
    time_idx = int(np.where(dates == target)[0][0])
    print(f'time_mode {time_mode} -> time index {time_idx}')
elif isinstance(time_mode, int):
    time_idx = time_mode
else:
    time_idx = None
n_times_expected = len(output_dates) if output_dates is not None else None

In [ ]:
results = []          # one row per case for the main CSV
results_per_time = [] # one row per case/fault with every time step
skipped = []
short_cases = [] # (case, n_times) for simulations that stopped early

for case_num in tqdm(case_nums, desc='Stress-based FSA'):
    files = {k: principal_stresses_folder/f'case{case_num}_{k}.npy' for k in ['STRESMXP','STRESMNP','STRESINT']}
    row_out = {'case': case_num}
    if not all(p.exists() for p in files.values()):
        skipped.append(case_num)
        for f in range(n_faults):
            row_out[f'fault{f}'] = np.nan
        results.append(row_out)
        continue

    # load principal stresses (float64 to avoid precision issues), normal faulting: Sv > SH > Sh
    Sv = np.load(files['STRESMXP']).astype(np.float64)
    Sh = np.load(files['STRESMNP']).astype(np.float64)
    SH = np.load(files['STRESINT']).astype(np.float64)
    if n_times_expected is not None and Sv.shape[-1] < n_times_expected:
        short_cases.append((case_num, Sv.shape[-1]))
    SH_azi = cases.loc[cases['case'] == case_num, 'SH_azi_deg'].iloc[0]

    for f in range(n_faults):
        m = fault_masks[f]
        # stresses passed already masked -> sigma, tau shape (n_fault_cells, n_times)
        sigma, tau, _ = FSA_stress_based_per_cell(
            SH = SH[m], Sh = Sh[m], Sv = Sv[m],
            SH_azi = SH_azi,
            fault_strike = fault_strikes[f],
            fault_dip = fault_dips[f],
            fault_cohesion = fault_cohesion,
            friction_coefficient = friction_coefficient)
        ratio = (tau - fault_cohesion) / sigma # (n_fault_cells, n_times)
        p_t = np.nanpercentile(ratio, percentile, axis=0)          # (n_times,)

        n_t = p_t.size # number of time steps this case reached
        if time_idx is not None:
            value = p_t[time_idx] if time_idx < n_t else np.nan # NaN if the simulation did not reach this time
        elif time_mode == 'max':
            value = np.nanmax(p_t[1:]) if n_t > 1 else p_t[0]
        else: # 'last'
            value = p_t[-1]
        row_out[f'fault{f}'] = value
        results_per_time.append({'case': case_num, 'fault_id': f, 'n_times': n_t, **{f'time_{t}': v for t, v in enumerate(p_t)}})
    results.append(row_out)

df = pd.DataFrame(results).sort_values('case').reset_index(drop=True)
df.to_csv(output_csv_path, index=False, float_format='%.6f')
pd.DataFrame(results_per_time).to_csv(output_csv_per_time_path, index=False, float_format='%.6f')

print(f'Saved {output_csv_path}')
print(f'Cases that stopped early (case, n_times reached): {short_cases}')
print(f'Cases skipped (missing/incomplete stress files, written as NaN): {skipped}')
df

## Quick check

In [ ]:
print(df.describe())
print('Cases with ratio >= friction coefficient:')
df[(df[[f'fault{f}' for f in range(n_faults)]] >= friction_coefficient).any(axis=1)]